# Palm post-processing I

This notebook takes in the ML detection output and generates point geolocations and palm census.

Import required libraries

In [1]:
# imports
import rasterio, pandas as pd, geopandas as gpd, numpy as np
from sklearn.neighbors import KDTree
from pathlib import Path

### Palm geolocation and census
The detected palms are used to generate a point position within the set boundary of the block.

In [ ]:
class Census:
    
    def __init__(self, in_tiff_path, in_csv_path, in_boundary_path, in_block_id):
        "in_tiff_path: tiled geotiff images"
        
        self.csv_path = Path(in_csv_path)
        self.image_path = Path(in_tiff_path)
        self.boundary_path = Path(in_boundary_path)
        self.block_id = in_block_id
        
        # input directory validation
        if not self.csv_path.exists() and self.image_path.exists():
            raise FileNotFoundError(f"one or both directories '{self.csv_path}', and '{self.image_path}' does not exist.")
            return
        
    
    # match number of image tiles to csv's
    def img_csv_match (self):        
        if len(list(self.csv_path.iterdir())) == len(list(self.image_path.iterdir())):
            
            # validate same items in both csv and image dir
            all_csvs = {csv.stem for csv in self.csv_path.iterdir()}
            all_imgs = {img.stem for img in self.image_path.iterdir()}

            # Assert that both sets contain the same items
            assert all_imgs == all_csvs, f"Items do not match. Missing image files: {all_csvs - all_imgs}, Missing CSV files: {all_imgs - all_csvs}"
       
        else: print('csv and image datasets do not have the same number of items.')
        
    
    def diameter(self, row):        
        # compute lat and long values
        lat = abs(row['b_max_x'] - row['b_min_x'])
        long = abs(row['b_max_y'] - row['b_min_y'])
        
        # get the lowest diameter range.
        cal_diameter = lat if lat < long else long

        if cal_diameter > 8:
            cal_diameter = np.random.uniform(7, 8)
        
        # get diameter to 2dp
        return round(cal_diameter, 2)


    def aoi_gdf(self, in_gdf, in_bdry_path):
        boundary = gpd.read_file(in_bdry_path)
        
        if in_gdf.crs != boundary.crs:
            boundary = boundary.to_crs(in_gdf.crs)
        
        clipped_gdf = gpd.clip(in_gdf, boundary)

        return clipped_gdf

    
    # read files create points
    def generate_geolocators (self, in_csv_file):        
        df = pd.read_csv(in_csv_file)

        # get the label as numbers
        df['Plant_id'] = list(range(len(df)))

        # get the center points
        center_y = df['keypoint_x_center']
        center_x = df['keypoint_y_center']
            
        # get boundary_pixel values
        bx_min, bx_max, by_min, by_max = df[['bbox_x_min', 'bbox_x_max', 'bbox_y_min', 'bbox_y_max']].values.T

        # get image extension and image file
        file_ext = [img_file.suffix for img_file in self.image_path.iterdir()]
        geotiff = f'{self.image_path}/{in_csv_file.stem}{file_ext[0]}'

        # convert to geo-coordinates
        with rasterio.open(geotiff) as image:
            img_crs = image.crs

            center = image.xy(center_x, center_y)
            box_min = image.xy(bx_min, by_min)
            box_max = image.xy(bx_max, by_max)

        # assign coordinates
        df['lat'], df['long'] = (center[1], center[0])
            
        df['b_min_x'], df['b_min_y'] = (box_min[0], box_min[1])
        df['b_max_x'], df['b_max_y'] = (box_max[0], box_max[1])
    
        df['diameter'] = df.apply(self.diameter, axis=1)

        # keep necessary files
        df = df[['Plant_id','long','lat','diameter']]

        # create geodataframe
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['long'], df['lat']), crs=img_crs)
        gdf = gdf.drop_duplicates(subset='geometry')

        return gdf
    
    
    # removes duplicate gdf points 
    def deduplicate_points_kdtree(self, gdf, dedup_distance=4.9):

        # Build KDTree
        coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
        tree = KDTree(coords)

        # Find all neighbors within dedup_distance
        indices = tree.query_radius(coords, r=dedup_distance)

        # Identify duplicates to drop
        to_drop = set()
        for i, neighbors in enumerate(indices):
            for j in neighbors:
                if i != j and j not in to_drop and i not in to_drop:
                    # Keep the lower index, drop the higher one
                    drop_idx = max(i, j)
                    to_drop.add(drop_idx)

        # Drop duplicates
        deduped_gdf = gdf.drop(gdf.index[list(to_drop)])

        return deduped_gdf


    def process(self):
        # check input csv and image file
        self.img_csv_match()

        # get all file points
        gdf_list = []

        # read csv's as df
        for csv_file in self.csv_path.iterdir():
            
            gdf = self.generate_geolocators(csv_file)

            # Append gdf to gdf_list
            gdf_list.append(gdf)
        
        # merge all gdf_list
        merged_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

        clipped_gdf = self.aoi_gdf(merged_gdf, self.boundary_path)

        # remove duplicates before assigning new_id to the plants
        clipped_gdf = self.deduplicate_points_kdtree(clipped_gdf)

        # assign new_id to the plants
        clipped_gdf['Plant_id'] = (clipped_gdf.index + 1).astype(str)

        # set block id
        clipped_gdf['block_id'] = self.block_id

        # create a point_loc
        points_path = self.image_path.resolve().parent.joinpath('count_geo_output')
            
        points_path.mkdir(parents=True, exist_ok=True)

        # Convert (reproject) to geographic coordinates
        gdf_geo = clipped_gdf.to_crs("EPSG:4326")

        # save the merged_df as geojson file
        clipped_gdf.to_file(f'{points_path}/model_points.geojson', driver='GeoJSON')

        # indicate points completion
        print(f'{points_path} successfully completed.')
            

# Data repository
input_image_dir= '.../Project_repo/count/count_image_tiles/' # imagery directory
input_csv_dir = '.../Project_repo/count/count_ML_output/' # model csv output directory
input_boundary_dir = '.../Project_repo/boundary_data/geojson_data/' # location of boundary data
input_block_id = 1 # set the block id

# Implementation
post_processing = Census(input_image_dir, input_csv_dir, input_boundary_dir,input_block_id)
post_processing.process()